In [ ]:
import numpy as np
import pyvista as pv
from pyau3d.files import TopFile, PartFile, ModeFile
from pyau3d.utils import PltFileUtils, NodFileUtils, AnimSmartStream, GrpFileUtils
from pyau3d.pv.loaders import GrpFileVTK
from pyau3d.pv.loader.grpfilevtk import _grp2pv
from pyau3d.pv.vtktool.pdata import polydata
import vtk
import matplotlib.pyplot as plt
from matplotlib import cm, ticker
from pyau3d.constants import Constants
# Enable interactive backend
%matplotlib widget
 

# specify group for animation file
# group = 2 corresponds to cylinder wall - could double check in .an02 file
RE = 50
GROUP = 2
 
cfd_dir = f"/home/ahf25/CFD_2d_cylinder_all/Steady/2d_cylinder_Re{RE}"
plt_file = "cylinder.plt"
part_file = "cylinder.part"

# specify file paths
path2plt = cfd_dir  + "/" +  plt_file
path2part = cfd_dir  + "/" +  part_file
 
mesh = PltFileUtils(path2plt)
part_file = PartFile(path2part)

 

In [47]:
side_coords, side_nodes, side_nodemap = mesh.extract_surface(flag=3)

coord = mesh.coord
ifac3 = mesh.ifac3


In [26]:

# write function to calculate area_normals
# not using polydata.area_normals since ifac4 is not available in my mesh

def area_normals(coord, ifac3):
    """
    Returns a pyvista.PolyData object with point normals scaled by the area of adjacent faces.

    Args:
        coord  : (N, 3) array of mesh point coordinates
        ifac3  : (M, 3) array of triangle connectivity (zero-based indices)

    Returns:
        pdata  : pyvista.PolyData with 'anor' field = area-weighted normals at each point
    """
    nfac3 = len(ifac3)

    # --- build PolyData from triangles ---
    faces = np.hstack([
        np.full((nfac3, 1), 3),   # each face has 3 vertices
        ifac3
    ]).flatten()

    pdata = pv.PolyData(coord, faces)

    # --- compute face areas ---
    pdata = pdata.compute_cell_sizes(length=False, volume=False)  # adds 'Area' field
    face_areas = pdata['Area']  # shape (nfac3,)

    # --- accumulate area contribution at each point ---
    # each triangle shares its area equally among its 3 vertices
    points_area = np.zeros(len(coord))
    np.add.at(points_area, ifac3, face_areas[:, np.newaxis] / 3.0)

    # --- scale point normals by accumulated area ---
    pdata['anor'] = -pdata.point_normals * points_area[:, np.newaxis]

    return pdata

In [48]:
blade = area_normals(coord, ifac3)
# # define delta t
# deltat = 0.001
# force = []
# # start of frame = 1
# # end of frame = 2000
# # interval = 1
# iframe = np.arange(1,2000,1)


# x = grp.coords[:,0]
# y = grp.coords[:,1]
# z = grp.coords[:,2]

# locp = np.searchsorted(nod.ivars[ianimgrp - 1],9)
# variables = np.arange(nod.nvars[ianimgrp - 1])

# data = anim.read_frame(i,variables, )#redim = True,)

# blade['p'] = data[:, locp] * 10000
# blade['F'] = blade['p'][:] * np.dot(blade['anor'],[0,1,0]) # compute lift force area normal * y_unit vector

# Force = np.sum(blade['F'], axis = 0)


CellSizeError: `faces` cell array size is invalid.

In [ ]:
U_inf = 34.02626486 # farfield velocity (m/s)
rho_inf = 2.10E-05 # farfield density (kg/m^3)
surface_area = 1

CL = force / (0.5 * rho_inf * U_inf **2 * surface_area)

print("CL: ", CL)